In [4]:
import gzip
import re
import pandas as pd
import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

# ============================================================
# USER SETTINGS
# ============================================================
gz_file = "hmc.forcing-grid.201805010000.nc.gz"
var_name = "Terrain"   # change to Rain, AirTemperature, Wind, etc.

out_tif = f"{var_name}_201805010000_correct.tif"
nodata_value = -9999

# ============================================================
# 1. OPEN NETCDF.GZ DIRECTLY AND FIX FAKE TIME
# ============================================================
with gzip.open(gz_file, "rb") as f:
    ds = xr.open_dataset(
        f,
        decode_times=False,
        engine="h5netcdf"
    )

# rename coordinates safely
rename_dict = {}

if "Longitude" in ds:
    rename_dict["Longitude"] = "lon"
if "Latitude" in ds:
    rename_dict["Latitude"] = "lat"
if "Y" in ds.dims:
    rename_dict["Y"] = "y"
if "X" in ds.dims:
    rename_dict["X"] = "x"

ds = ds.rename(rename_dict)

# get correct datetime from filename
match = re.search(r"\d{12}", gz_file)
dt = pd.to_datetime(match.group(), format="%Y%m%d%H%M")

if "time" in ds.coords or "time" in ds.dims:
    ds = ds.assign_coords(time=[dt])

print(ds)

# ============================================================
# 2. READ VARIABLE AND COORDINATES
# ============================================================
data_nc = ds[var_name].values

# remove time dimension if variable has time
if data_nc.ndim == 3:
    data_nc = data_nc[0, :, :]

lat = ds["lat"].values
lon = ds["lon"].values

print("NetCDF data shape:", data_nc.shape)
print("Latitude shape:", lat.shape)
print("Longitude shape:", lon.shape)

print("NetCDF lon range:", np.nanmin(lon), np.nanmax(lon))
print("NetCDF lat range:", np.nanmin(lat), np.nanmax(lat))

# ============================================================
# 3. CREATE CORRECT GEOTIFF
# ============================================================
dx = abs(lon[0, 1] - lon[0, 0])
dy = abs(lat[1, 0] - lat[0, 0])

west = np.nanmin(lon) - dx / 2
north = np.nanmax(lat) + dy / 2

transform = from_origin(west, north, dx, dy)

# IMPORTANT: flip vertically for GeoTIFF
data_tif = np.flipud(data_nc)

with rasterio.open(
    out_tif,
    "w",
    driver="GTiff",
    height=data_tif.shape[0],
    width=data_tif.shape[1],
    count=1,
    dtype=data_tif.dtype,
    crs="EPSG:4326",
    transform=transform,
    nodata=nodata_value
) as dst:
    dst.write(data_tif, 1)

print("Saved TIFF:", out_tif)

# ============================================================
# 4. READ TIFF BACK FOR VERIFICATION
# ============================================================
with rasterio.open(out_tif) as src:
    tif_read = src.read(1)
    bounds = src.bounds
    crs = src.crs
    transform_read = src.transform

print("TIFF CRS:", crs)
print("TIFF bounds:", bounds)
print("TIFF transform:", transform_read)

# flip TIFF back to NetCDF orientation for numerical check
tif_back_to_nc = np.flipud(tif_read)

difference = tif_back_to_nc - data_nc

difference_plot = np.where(data_nc == nodata_value, np.nan, difference)

print("Maximum absolute difference:",
      np.nanmax(np.abs(difference_plot)))

# ============================================================
# 5. PLOT ORIGINAL NETCDF
# ============================================================
data_nc_plot = np.where(data_nc == nodata_value, np.nan, data_nc)

plt.figure(figsize=(9, 6))
plt.pcolormesh(lon, lat, data_nc_plot, shading="auto")
plt.colorbar(label=var_name)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"Original NetCDF: {var_name}")
plt.tight_layout()
plt.show()

# ============================================================
# 6. PLOT CONVERTED TIFF
# ============================================================
tif_plot = np.where(tif_read == nodata_value, np.nan, tif_read)

plt.figure(figsize=(9, 6))
plt.imshow(
    tif_plot,
    extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
    origin="upper"
)
plt.colorbar(label=var_name)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title(f"Converted GeoTIFF: {var_name}")
plt.tight_layout()
plt.show()

# ============================================================
# 7. PLOT DIFFERENCE
# ============================================================
plt.figure(figsize=(9, 6))
plt.pcolormesh(lon, lat, difference_plot, shading="auto")
plt.colorbar(label="TIFF - NetCDF")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Difference after re-aligning TIFF to NetCDF")
plt.tight_layout()
plt.show()

<xarray.Dataset> Size: 7MB
Dimensions:         (time: 1, y: 206, x: 446)
Coordinates:
  * time            (time) datetime64[ns] 8B 2018-05-01
    lon             (y, x) float64 735kB ...
    lat             (y, x) float64 735kB ...
Dimensions without coordinates: y, x
Data variables:
    crs             (time) int32 4B ...
    Terrain         (y, x) float64 735kB ...
    AirTemperature  (y, x) float64 735kB ...
    Rain            (y, x) float64 735kB ...
    IncRadiation    (y, x) float64 735kB ...
    Wind            (y, x) float64 735kB ...
    RelHumidity     (y, x) float64 735kB ...
    LAI             (y, x) float64 735kB ...
Attributes: (12/27)
    comment:            Author(s): Daniele Dolia; Simone Gabellani; Fabio Delogu
    project:            FloodPRoofs VdA
    references:         http:cf-pcmdi.llnl.gov/; http:Fcf-pcmdi.llnl.gov/docu...
    website:            http:cimafoundation.org
    institution:        CIMA Research Foundation - www.cimafoundation.org
    algorithm:  

ValueError: I/O operation on closed file